In [1]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities_copy import extractor
import uproot
import awkward as ak    

x_MH125=extractor("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH125_LH_2017.root", "Events")


file=uproot.open("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH125_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
booleanas=tree.arrays(["FatJet_isMatchedWith2BHadrons"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]
Fatjet_isMatchedWith2BHadrons= booleanas["FatJet_isMatchedWith2BHadrons"]
#Filtriamo i dati

mask = (ak.flatten(Fatjet_isMatchedWithA) == 1) & (ak.flatten(Fatjet_isMatchedWith2BHadrons) == 1)
x_filtered = x_MH125[mask]


/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/riccardo/anaconda3/envs/rootnev/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "
/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


In [2]:
from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()
x_easy=[x for x in x_plot if 75 < x < 175]

def voigt2(x, norm, mu, sigma, gamma, norm2, mu2, sigma2, gamma2):
    return voigt_profile(x-mu, sigma, gamma) * norm + norm2*voigt_profile(x-mu2, sigma2, gamma2)

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_easy) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_easy) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt2)

m_voigt=Minuit(ls_voigt,  norm=1, mu=125, sigma=5, gamma=1, norm2=1, mu2=100, sigma2=5, gamma2=0.001)
m_voigt.limits["mu"]= (100, 150)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.fixed["gamma2"]= True

m_voigt.migrad()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 63.45 (χ²/ndof = 1.5)      │              Nfcn = 672              │
│ EDM = 0.000101 (Goal: 0.0002)    │            time = 0.2 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name   │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm   │   0.804   │   0.029   │            │            │         │         │       │
│ 1 │ mu     │  128.52   │   0.13    │            │            │   100   │   150   │       │
│ 2 │ sigma  │   7.02    │   0.24    │            │            │   0.1   │   20    │       │
│ 3 │ gamma  │   4.36    │   0.19    │            │            │  0.01   │   10    │       │
│ 4 │ norm2  │   0.241   │   0.028   │            │            │         │         │       │
│ 5 │ mu2    │   118.2   │    1.1    │            │            │         │         │       │
│ 6 │ sigma2 │   15.4    │    0.5    │            │            │         │         │       │
│ 7 │ gamma2 │  1.00e-3  │  0.01e-3  │            │            │         │         │  yes  │
└───┴────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌────────┬─────────────────────────────────────────────────────────────────────────┐
│        │     norm       mu    sigma    gamma    norm2      mu2   sigma2   gamma2 │
├────────┼─────────────────────────────────────────────────────────────────────────┤
│   norm │ 0.000862  -0.6e-3   4.2e-3   0.1e-3  -0.8e-3 -31.3e-3  -7.2e-3        0 │
│     mu │  -0.6e-3   0.0164   -0.012    0.008   0.6e-3   -0.009   -0.035    0.000 │
│  sigma │   4.2e-3   -0.012   0.0558    -0.03  -4.3e-3    -0.16     0.01     0.00 │
│  gamma │   0.1e-3    0.008    -0.03   0.0358   0.2e-3     0.01    -0.05     0.00 │
│  norm2 │  -0.8e-3   0.6e-3  -4.3e-3   0.2e-3 0.000757  29.7e-3   6.4e-3        0 │
│    mu2 │ -31.3e-3   -0.009    -0.16     0.01  29.7e-3      1.3     0.33      0.0 │
│ sigma2 │  -7.2e-3   -0.035     0.01    -0.05   6.4e-3     0.33    0.236     0.00 │
│ gamma2 │        0    0.000     0.00     0.00        0      0.0     0.00        0 │
└────────┴─────────────────────────────────────────────────────────────────────────┘

In [3]:
fit_MH125_values={}
fit_MH125_errors={}

fit_values={'MH125': fit_MH125_values,}
fit_errors={'MH125_errors': fit_MH125_errors}



for param in m_voigt.parameters:
    fit_MH125_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deveestarre gli errori 
    fit_MH125_errors[error] = m_voigt.errors[error]

print(fit_MH125_values)
print(fit_MH125_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH125"]=fit_MH125_values
results["MH125_errors"]=fit_MH125_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH125"]=fit_MH125_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH125_errors"]=fit_MH125_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  

{'norm': 0.8039950948218383, 'mu': 128.51733926756873, 'sigma': 7.022259215523558, 'gamma': 4.364438443343952, 'norm2': 0.24115382221527426, 'mu2': 118.2208304290723, 'sigma2': 15.36444119707335, 'gamma2': 0.001}
{'norm': 0.029356427314279634, 'mu': 0.12789214810176475, 'sigma': 0.23615433387630658, 'gamma': 0.18907607680284366, 'norm2': 0.027509106529394953, 'mu2': 1.141691835966887, 'sigma2': 0.48567091635428955, 'gamma2': 1e-05}
